In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split
from transformers import DistilBertModel, DistilBertTokenizerFast
from torch.utils.data import DataLoader
from torch.optim import AdamW

# PREP THE DATASET

In [5]:
!wget https://raw.githubusercontent.com/kyuz0/llm-chronicles/main/datasets/restaurant_reviews.csv

--2026-08-21 11:41:39--  https://raw.githubusercontent.com/kyuz0/llm-chronicles/main/datasets/restaurant_reviews.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2861025 (2.7M) [text/plain]
Saving to: ‘restaurant_reviews.csv’

restaurant_reviews. 100%[===================>]   2.73M  --.-KB/s    in 0.04s   

2026-08-21 11:41:39 (64.5 MB/s) - ‘restaurant_reviews.csv’ saved [2861025/2861025]



In [6]:
df=pd.read_csv('restaurant_reviews.csv')
df.head()

,Review,Rating
0,The ambience was good food was quite good . ha...,positive
1,Ambience is too good for a pleasant evening. S...,positive
2,A must try.. great food great ambience. Thnx f...,positive
3,Soumen das and Arun was a great guy. Only beca...,positive
4,Food is good.we ordered Kodi drumsticks and ba...,positive


In [8]:
df.Rating.value_counts()

,count
Rating,
positive,6331
negative,2428
neutral,1192


In [9]:
sentiment_mapping={"positive":0, "negative":1,"neutral":3}

In [10]:
df.Rating=df.Rating.map(sentiment_mapping)

In [12]:
# Display the first few rows of the dataframe
print(df.head())

# Display statistics about the dataset
print("\nDataset Statistics:")
print(df['Rating'].value_counts())

                                              Review  Rating
0  The ambience was good food was quite good . ha...       0
1  Ambience is too good for a pleasant evening. S...       0
2  A must try.. great food great ambience. Thnx f...       0
3  Soumen das and Arun was a great guy. Only beca...       0
4  Food is good.we ordered Kodi drumsticks and ba...       0

Dataset Statistics:
Rating
0    6331
1    2428
3    1192
Name: count, dtype: int64


In [13]:
class CustomDatset(Dataset):
  def __init__(self,csv,tokenizer,max_length):
    self.dataset = pd.read_csv(csv)
    self.tokenizer = tokenizer
    self.max_length = max_length
    self.sentiment_mapping={"positive":0, "negative":1,"neutral":3}


  def __len__(self):
    return len(self.dataset)

  def __getitem__(self,idx):
    review_text = self.dataset.loc[idx, 'Review']
    sentiment = self.dataset.loc[idx, 'Rating']
    labels=self.sentiment_mapping[sentiment]

    encoding=self.tokenizer.encode_plus(
          review_text,
          add_special_tokens=True,  # Add [CLS] token at the start for classification
          max_length=self.max_length,
          return_token_type_ids=False,
          padding='max_length',
          return_attention_mask=True,
          return_tensors='pt',
          truncation=True
        )

    return {
        'review_text': review_text,
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(labels, dtype=torch.long)
    }


